In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


# Figure 6 plotting code


## Shared setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from scipy.spatial import ConvexHull, QhullError

ROOT = Path.cwd()
DATA = ROOT / "source_data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
assert DATA.exists(), "Run this notebook from the Figure 6 code directory."

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

blue = "#2B5C96"
red = "#C94C4C"
grey = "#8B9AA7"
dark = "#152536"

cell_colors = {
    "Epithelial": "#1F7A7A", "Fibroblast": "#E59B3A", "Endothelial": "#39A56A",
    "Pericyte": "#93B5D7", "T cell": "#E64B35", "NK cell": "#7E57C2",
    "B cell": "#2C7FB8", "Plasma cell": "#4EA3D8", "Monocyte/Macrophage": "#61C2A2",
    "Dendritic cell": "#D94F9B", "Mast cell": "#8CD36B", "Neutrophil": "#37BDBD",
    "Mix": "#C9C9C9", "Unknown": "#D9D9D9",
    "T/NK cell": "#4FA366", "Myeloid": "#EF6C42", "Fibroblast/FDC": "#F1A55C",
    "Endothelial/Pericyte": "#5EB4A5", "Other/Unknown": "#BDBDBD", "Other immune": "#BDBDBD",
    "B cells": "#2C7FB8", "DC": "#D94F9B", "DC cells": "#D94F9B",
    "Endothelial cells": "#39A56A", "Epithelial cells": "#1F7A7A",
    "Fibroblasts": "#E59B3A", "Macrophages": "#61C2A2",
    "Mast cells": "#8CD36B", "Monocytes": "#61C2A2",
    "T cells": "#E64B35", "NK cells": "#7E57C2"
}

class_colors = {"Conforming TLS": blue, "Deviating TLS": red, "Mature TLS": "#2B7A3D"}

from matplotlib.gridspec import GridSpec
def savefig(fig, name):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{name}.{ext}", dpi=450, bbox_inches="tight")

def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def draw_hull(ax, df, x="X", y="Y", color="k", lw=0.55, alpha=1, pad=0):
    pts = df[[x, y]].dropna().drop_duplicates().to_numpy()
    if len(pts) < 3:
        return
    try:
        h = ConvexHull(pts)
        poly = Polygon(pts[h.vertices], closed=True, fill=False, ec=color, lw=lw, alpha=alpha, joinstyle="round")
        ax.add_patch(poly)
    except QhullError:
        pass

def add_panel(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")


## Fig. 6b


In [ ]:
def panel_b(ax):
    s3 = pd.read_csv(DATA / "slide_seq_annotations" / "S3_final_annotation_with_TLS.csv.gz")
    order = [c for c in cell_colors if c in set(s3["celltype_2_ZZM"])]
    for ct in order:
        sub = s3[s3["celltype_2_ZZM"] == ct]
        ax.scatter(sub["X"], sub["Y"], s=0.8, c=cell_colors[ct], lw=0, rasterized=True, label=f"{ct} ({len(sub):,})")
    tls = s3[s3["tls_region_id_ZZM"] > 0]
    for _, sub in tls.groupby("tls_region_id_ZZM"):
        draw_hull(ax, sub, color="#333333", lw=0.45)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("S3 Slide-seq annotation and TLS-like regions", loc="left", pad=2)
    ax.legend(markerscale=5, fontsize=5, bbox_to_anchor=(1.02, 1.0), loc="upper left", ncol=1, handletextpad=0.2)

family_colors = {"PA/DAG/PI signaling": "#B6424B", "Phospholipid remodeling": "#E06A4B", "SM/CE membrane lipids": "#7667A9", "TG/fatty-acyl stress": "#326D79", "other DESI features": "#DCE9EA"}


## Fig. 6c


In [ ]:
def panel_c(ax):
    desi = pd.read_csv(DATA / "SourceData_Fig6_DESI_EPAS1high_vs_low_features.csv")
    desi["family"] = desi["selected_lipid_family"].fillna("other DESI features")
    desi["x"] = desi["delta_z_high_minus_low"]
    desi["y"] = desi["neglog10_p"]
    ax.scatter(desi["x"], desi["y"], s=5, c=family_colors["other DESI features"], lw=0, alpha=0.5, rasterized=True)
    sel = desi[desi["selected_lipid_program_feature"] == True].copy()
    for fam, sub in sel.groupby("family"):
        ax.scatter(sub["x"], sub["y"], s=34, c=family_colors.get(fam, "#777777"), edgecolor="white", lw=0.4, label=fam, zorder=3)
    for _, r in sel.iterrows():
        if pd.notna(r.get("short_label")):
            ax.annotate(r["short_label"], (r["x"], r["y"]), xytext=(4, 4), textcoords="offset points", fontsize=5,
                        arrowprops=dict(arrowstyle="-", lw=0.3, color="#667"))
    ax.axhline(-np.log10(0.05), color="#6F91A0", ls="--", lw=0.6)
    ax.axvline(0, color="#9DB5C0", lw=0.6)
    ax.set_xlabel("mean delta z-score (EPAS1-high - EPAS1-low)")
    ax.set_ylabel("-log10 Welch P")
    ax.set_title("EPAS1-associated DESI lipid features", loc="left", pad=2)
    ax.legend(fontsize=5, loc="upper right", handletextpad=0.3)

def plot_region(ax, data, value=None, cmap="Reds", vlim=None):
    if value is None:
        for ct, sub in data.groupby("broad_celltype"):
            ax.scatter(sub["x"], sub["y"], s=9, c=cell_colors.get(ct, "#BDBDBD"), lw=0, rasterized=True)
    else:
        kwargs = dict(c=data[value], cmap=cmap, s=9, lw=0, rasterized=True)
        if vlim is not None:
            kwargs.update(vmin=vlim[0], vmax=vlim[1])
        ax.scatter(data["x"], data["y"], **kwargs)
    draw_hull(ax, data.rename(columns={"x": "X", "y": "Y"}), color="#333333", lw=0.55)
    ax.set_aspect("equal"); ax.axis("off")


## Fig. 6d


In [ ]:
def panel_d(fig, outer):
    spot = pd.read_csv(DATA / "SourceData_Fig6_spot_EPAS1_lipid_program.csv")
    rows = [("S8", 29, "Deviating TLS"), ("S8", 32, "Conforming TLS")]
    subgs = outer.subgridspec(2, 3, hspace=0.12, wspace=0.08)
    titles = ["Cell type", "EPAS1", "DESI lipid program"]
    axes=[]
    for i, (sid, rid, label) in enumerate(rows):
        d = spot[(spot["sample"] == sid) & (spot["struct_rid"] == rid)].copy()
        for j in range(3):
            ax = fig.add_subplot(subgs[i, j]); axes.append(ax)
            if i == 0:
                ax.set_title(titles[j], pad=2)
            if j == 0:
                plot_region(ax, d)
                ax.text(-0.08, 0.5, label, transform=ax.transAxes, rotation=90, ha="right", va="center", color=class_colors.get(label, dark), fontsize=6)
                ax.text(0.02, 0.02, f"{sid}_TLS_{rid}", transform=ax.transAxes, ha="left", va="bottom", fontsize=5)
            elif j == 1:
                plot_region(ax, d, "EPAS1_z", "Reds", (-1, 3))
            else:
                plot_region(ax, d, "lipid_program_z", "RdYlBu_r", (-3, 3))
    axes[0].text(-0.16, 1.18, "d", transform=axes[0].transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")
    axes[0].text(0.03, 1.18, "Spatial multi-omics view of EPAS1-high TLS lipid remodelling", transform=axes[0].transAxes, ha="left", va="bottom", fontsize=8)


## Fig. 6f


In [ ]:
def panel_f(ax1, ax2):
    bcr = pd.read_csv(DATA / "SourceData_Fig6_Bcell_BCR_umi2_sample_metrics.csv")
    metrics = [("bcr_detection_percent", "BCR detection", "B cells (%)"), ("expanded_clone_percent", "Expanded clones (>=2)", "BCR+ cells (%)")]
    for ax, (metric, title, ylabel) in zip([ax1, ax2], metrics):
        vals = bcr.pivot(index="sample", columns="score_group", values=metric)
        means = vals[["score-low", "score-high"]].mean()
        errs = vals[["score-low", "score-high"]].apply(sem)
        ax.bar([0,1], means, yerr=errs, color=[grey, red], edgecolor="white", lw=0.5, capsize=2, width=0.65)
        for i, lab in enumerate(["score-low", "score-high"]):
            ax.text(i, means[lab]*0.78, f"{'Low' if i==0 else 'High'}\n{means[lab]:.0f}%", ha="center", va="center", fontsize=6, fontweight="bold", color="white" if i==1 else dark)
        ax.set_xticks([0,1]); ax.set_xticklabels(["Low", "High"])
        ax.set_title(title, pad=2); ax.set_ylabel(ylabel); ax.set_ylim(0, 110)
    add_panel(ax1, "f")


## Fig. 6h


In [ ]:
def p_lookup(stats, metric, comparison):
    sub = stats[(stats["scope"] == "All cells") & (stats["metric"] == metric) & (stats["comparison"] == comparison)]
    if sub.empty:
        return None
    return float(sub.iloc[0]["p_paired_t"])

def p_label(p):
    if p is None or not np.isfinite(p): return ""
    return "P<0.001" if p < 0.001 else f"P={p:.2g}"

def bracket(ax, x1, x2, y, text):
    ax.plot([x1,x1,x2,x2], [y,y+0.01,y+0.01,y], color=dark, lw=0.5)
    ax.text((x1+x2)/2, y+0.012, text, ha="center", va="bottom", fontsize=5)

def panel_h(ax):
    df = pd.read_csv(DATA / "SourceData_Fig6_scCellFie_pseudobulk_by_donor_scope.csv")
    stats = pd.read_csv(DATA / "SourceData_Fig6_scCellFie_key_question_paired_stats.csv")
    metric = "Hypoxia-up 9-task score"
    d = df[df["scope"] == "All cells"].copy()
    order = ["S5", "S1", "S2", "S4", "S3"]
    labels = ["G1\nBlank", "G2\nNo drug", "G3\nBelzutifan", "G4\nFG-4592", "G5\nBelzutifan\n+ FG-4592"]
    colors = ["#536B7A", blue, "#48A172", "#F49B38", "#C56AA5"]
    ax.axvspan(2.6, 4.4, color="#FBE8DF", zorder=0)
    for donor, sub in d.groupby("donor"):
        y = [sub[sub["big_sample"] == s][metric].iloc[0] if (sub["big_sample"] == s).any() else np.nan for s in order]
        ax.plot(range(len(order)), y, color="#CFCFCF", lw=0.8, marker="o", ms=2.5, mfc="white", mec="#9CA6AF", zorder=1)
    for i, s in enumerate(order):
        vals = d[d["big_sample"] == s][metric]
        ax.errorbar(i, vals.mean(), yerr=sem(vals), fmt="o", ms=5.5, color=colors[i], mec=dark, mew=0.5, capsize=2, zorder=3)
    ymax = d[metric].max()
    bracket(ax, 1, 2, ymax + 0.025, p_label(p_lookup(stats, metric, "Belzutifan_alone_vs_NoDrug")))
    bracket(ax, 1, 3, ymax + 0.075, p_label(p_lookup(stats, metric, "FG4592_vs_NoDrug")))
    bracket(ax, 3, 4, ymax + 0.05, p_label(p_lookup(stats, metric, "Belzutifan_effect_under_FG4592")))
    ax.set_xticks(range(len(order))); ax.set_xticklabels(labels)
    ax.set_ylabel("scCellFie pseudobulk score")
    ax.set_title(metric, pad=2)
    add_panel(ax, "h")
